# 04 · Valor de negocio para LATAM
Línea base, efecto de la alerta en minutos y US$ por escenario, sensibilidad del umbral, hallazgos accionables y tabla de trazabilidad (objetivo, hallazgo, decisión, responsable, efecto). Los supuestos están en `config/pipeline.yaml` (bloque `valor_negocio`).

In [ ]:
import os, sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
os.environ.setdefault("LATAM_MODO", "local")          # "aws" en el clúster (lee s3a://$LAKE_BUCKET)
os.environ.setdefault("LATAM_LAKE_LOCAL", str(RAIZ / "data" / "lake"))
import pandas as pd
import matplotlib.pyplot as plt
from latam_ops.comun.config import cargar_yaml, ruta_repo
VALOR = ruta_repo("docs", "evidencias", "valor")
if not (VALOR / "valor_escenarios.csv").exists():
    from latam_ops.ml import valor_negocio
    valor_negocio.ejecutar()
cargar_yaml("pipeline")["valor_negocio"]["escenarios"]

## Línea base
Tasa de salidas LATAM con 15 y 60 min o más en el último año completo (ANAC VRA) y OTP de llegada publicado por Cirium.

In [ ]:
pd.read_csv(VALOR / "linea_base.csv").T

## Efecto de la alerta por escenario
VP = demoras avisadas a tiempo, FP = falsas alarmas. El ahorro usa el costo por minuto de A4A (2025) y la fracción de demora que se evita al actuar a T-2 h (supuesto).

In [ ]:
valor = pd.read_csv(VALOR / "valor_escenarios.csv")
valor[["objetivo", "escenario", "umbral", "alertas", "vp", "fp", "minutos_evitados", "ahorro_usd", "costo_alertas_usd",
       "beneficio_neto_usd", "vuelos_recuperados", "tasa_base", "tasa_tras_mitigar", "mejora_puntos_porcentuales"]]

In [ ]:
v15 = valor[valor["objetivo"] == "y_demora15"].set_index("escenario")
ax = v15[["ahorro_usd", "costo_alertas_usd", "beneficio_neto_usd"]].plot.bar(figsize=(8, 4))
ax.set_ylabel("US$ en el periodo evaluado"); ax.set_title("Alerta ámbar (15 min o más): ahorro, costo y beneficio neto")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## Sensibilidad del umbral (C_FN/C_FP)
Umbrales elegidos en validación y medidos en el periodo evaluado.

In [ ]:
pd.read_csv(VALOR / "sensibilidad_umbral.csv")

## Hallazgos accionables

In [ ]:
for nombre in ("aeropuertos", "franjas", "clima", "red"):
    print(f"\n== {nombre}")
    display(pd.read_csv(VALOR / f"hallazgos_{nombre}.csv"))

In [ ]:
clima = pd.read_csv(VALOR / "hallazgos_clima.csv").set_index("grupo")
ax = (clima["tasa_demora15"] * 100).sort_values().plot.barh(figsize=(7, 4))
ax.set_xlabel("% de salidas con 15 min o más"); ax.set_title("Condición en el METAR del origen a T-2 h")
plt.tight_layout(); plt.show()

## Trazabilidad del valor (sección 6.1)

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.read_csv(VALOR / "trazabilidad.csv")

## Supuestos y fuentes

In [ ]:
cfg = cargar_yaml("pipeline")["valor_negocio"]
print(cfg["fuente_costo_minuto"]); print(); print(cfg["linea_base"]["fuente_otp"]); print(); print(cfg["nota_supuestos"])